In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.9 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
from collections import defaultdict
from difflib import SequenceMatcher
import unicodedata

class TamilInscriptionAnalyzer:
    def __init__(self, excel_file_path):
        """
        Initialize with dictionary data from Excel file
        excel_file_path: path to Excel file with columns 'Old_Tamil', 'New_Tamil', 'Meaning'
        """
        self.dictionary = self._load_excel_dictionary(excel_file_path)
        self.word_index = self._build_word_index()
        print(f"Loaded {len(self.dictionary)} dictionary entries")

    def _load_excel_dictionary(self, file_path):
        """Load dictionary from Excel file"""
        try:
            df = pd.read_excel(file_path)

            # Handle different possible column names
            column_mapping = {
                'Old_Tamil': ['Old_Tamil', 'old_tamil', 'Old Tamil', 'OldTamil'],
                'New_Tamil': ['New_Tamil', 'new_tamil', 'New Tamil', 'NewTamil', 'Modern_Tamil', 'modern_tamil'],
                'Meaning': ['Meaning', 'meaning', 'English', 'english', 'Definition', 'definition']
            }

            # Find correct column names
            actual_columns = {}
            for standard_name, possible_names in column_mapping.items():
                for possible_name in possible_names:
                    if possible_name in df.columns:
                        actual_columns[standard_name] = possible_name
                        break

                if standard_name not in actual_columns:
                    actual_columns[standard_name] = None

            # Convert to dictionary format
            dictionary = []
            for _, row in df.iterrows():
                entry = {}

                if actual_columns['Old_Tamil']:
                    old_tamil = str(row[actual_columns['Old_Tamil']]) if pd.notna(row[actual_columns['Old_Tamil']]) else ''
                    entry['old_tamil'] = old_tamil.strip()
                else:
                    entry['old_tamil'] = ''

                if actual_columns['New_Tamil']:
                    new_tamil = str(row[actual_columns['New_Tamil']]) if pd.notna(row[actual_columns['New_Tamil']]) else ''
                    entry['modern_tamil'] = new_tamil.strip()
                else:
                    entry['modern_tamil'] = ''

                if actual_columns['Meaning']:
                    meaning = str(row[actual_columns['Meaning']]) if pd.notna(row[actual_columns['Meaning']]) else ''
                    entry['english'] = meaning.strip()
                else:
                    entry['english'] = ''

                # Only add entries with at least old Tamil text
                if entry['old_tamil'] and entry['old_tamil'] != 'nan':
                    dictionary.append(entry)

            return dictionary

        except Exception as e:
            print(f"Error loading Excel file: {e}")
            raise

    def _build_word_index(self):
        """Build an index for faster word lookup - includes both Old and New Tamil"""
        index = defaultdict(list)
        for i, entry in enumerate(self.dictionary):
            # Index Old Tamil words
            old_word = self._normalize_tamil(entry.get('old_tamil', ''))
            if old_word:
                index[len(old_word)].append((old_word, i, 'old'))
                index[f"first_{old_word[0]}"].append((old_word, i, 'old'))

            # Index New/Modern Tamil words
            new_word = self._normalize_tamil(entry.get('modern_tamil', ''))
            if new_word and new_word != old_word:  # Avoid duplicates
                index[len(new_word)].append((new_word, i, 'modern'))
                index[f"first_{new_word[0]}"].append((new_word, i, 'modern'))

        return index

    def _normalize_tamil(self, text):
        """Normalize Tamil text for better matching"""
        if not text or text == 'nan':
            return ""
        normalized = unicodedata.normalize('NFKD', str(text))
        cleaned = re.sub(r'[^\u0B80-\u0BFF]', '', normalized)
        return cleaned.strip()

    def _calculate_similarity(self, str1, str2):
        """Calculate similarity between two strings"""
        if not str1 or not str2:
            return 0.0

        similarity = SequenceMatcher(None, str1, str2).ratio()

        if str1 == str2:
            return 1.0

        # Bonus for substring matches
        if str1 in str2 or str2 in str1:
            shorter = min(str1, str2, key=len)
            longer = max(str1, str2, key=len)
            containment_bonus = len(shorter) / len(longer) * 0.3
            similarity = min(1.0, similarity + containment_bonus)

        # Length penalty
        len_diff = abs(len(str1) - len(str2))
        max_len = max(len(str1), len(str2))
        if max_len > 0:
            length_penalty = len_diff / max_len * 0.1
            similarity = max(0, similarity - length_penalty)

        return similarity

    def _find_candidate_words(self, text_segment, min_similarity=0.6):
        """Find candidate words that might match the text segment"""
        normalized_segment = self._normalize_tamil(text_segment)
        candidates = []

        if not normalized_segment:
            return candidates

        # Look for words of similar length
        target_len = len(normalized_segment)
        for check_len in range(max(1, target_len - 3), target_len + 4):
            if check_len in self.word_index:
                for word, dict_index, word_type in self.word_index[check_len]:
                    similarity = self._calculate_similarity(normalized_segment, word)
                    if similarity >= min_similarity:
                        candidates.append((word, dict_index, similarity, word_type))

        # Check words with same first character
        if normalized_segment:
            first_char_key = f"first_{normalized_segment[0]}"
            if first_char_key in self.word_index:
                for word, dict_index, word_type in self.word_index[first_char_key]:
                    similarity = self._calculate_similarity(normalized_segment, word)
                    if similarity >= min_similarity:
                        candidates.append((word, dict_index, similarity, word_type))

        # Remove duplicates and sort by similarity
        seen = set()
        unique_candidates = []
        for word, dict_index, similarity, word_type in candidates:
            if (dict_index, word_type) not in seen:
                seen.add((dict_index, word_type))
                unique_candidates.append((word, dict_index, similarity, word_type))

        return sorted(unique_candidates, key=lambda x: x[2], reverse=True)

    def segment_text(self, continuous_text, max_word_length=20, min_word_length=2):
        """Segment continuous text into potential words"""
        normalized_text = self._normalize_tamil(continuous_text)
        n = len(normalized_text)

        if n == 0:
            return []

        # Dynamic programming for optimal segmentation
        dp = [(0, [])] * (n + 1)

        for i in range(1, n + 1):
            for j in range(max(0, i - max_word_length), i):
                if i - j < min_word_length:
                    continue

                segment = normalized_text[j:i]
                candidates = self._find_candidate_words(segment, min_similarity=0.4)

                if candidates:
                    best_similarity = candidates[0][2]
                    word_score = best_similarity * (i - j)
                    total_score = dp[j][0] + word_score

                    if total_score > dp[i][0]:
                        dp[i] = (total_score, dp[j][1] + [(j, i, segment, candidates[0])])
                else:
                    fallback_score = dp[j][0] + (i - j) * 0.1
                    if fallback_score > dp[i][0]:
                        no_match = (segment, -1, 0.0, 'none')
                        dp[i] = (fallback_score, dp[j][1] + [(j, i, segment, no_match)])

        return dp[n][1] if dp[n][1] else self._fallback_segmentation(normalized_text)

    def _fallback_segmentation(self, text, chunk_size=4):
        """Fallback segmentation when dynamic programming fails"""
        segments = []
        for i in range(0, len(text), chunk_size):
            end = min(i + chunk_size, len(text))
            segment = text[i:end]
            candidates = self._find_candidate_words(segment, min_similarity=0.2)
            best_candidate = candidates[0] if candidates else (segment, -1, 0.0, 'none')
            segments.append((i, end, segment, best_candidate))
        return segments

    def analyze_inscription(self, continuous_text, min_similarity=0.5):
        """Main function to analyze continuous inscription text"""
        print(f"Analyzing text: {continuous_text[:50]}...")

        segments = self.segment_text(continuous_text)
        results = []
        unidentified_segments = []

        for start, end, segment, best_match in segments:
            word, dict_index, similarity, word_type = best_match

            if dict_index >= 0 and similarity >= min_similarity:
                dict_entry = self.dictionary[dict_index]
                result = {
                    'position': (start, end),
                    'original_segment': segment,
                    'matched_word': word,
                    'similarity': round(similarity, 3),
                    'match_type': word_type,
                    'old_tamil': dict_entry.get('old_tamil', ''),
                    'modern_tamil': dict_entry.get('modern_tamil', ''),
                    'english': dict_entry.get('english', ''),
                    'confidence': 'high' if similarity > 0.8 else 'medium' if similarity > 0.65 else 'low'
                }
                results.append(result)
            else:
                unidentified_segments.append({
                    'position': (start, end),
                    'segment': segment,
                    'best_similarity': round(similarity, 3) if dict_index >= 0 else 0.0
                })

        return {
            'original_text': continuous_text,
            'identified_words': results,
            'unidentified_segments': unidentified_segments,
            'total_segments': len(segments),
            'identification_rate': len(results) / len(segments) if segments else 0
        }

    def print_results(self, result):
        """Print analysis results"""
        print(f"\nOriginal text: {result['original_text']}")
        print(f"Identification rate: {result['identification_rate']:.2%}")
        print(f"Identified words: {len(result['identified_words'])}")

        print("\nIDENTIFIED WORDS:")
        print("=" * 50)

        for i, word in enumerate(result['identified_words'], 1):
            print(f"{i}. '{word['original_segment']}' → {word['matched_word']} [{word['match_type']}]")
            print(f"   Old: {word['old_tamil']} | Modern: {word['modern_tamil']}")
            print(f"   English: {word['english']}")
            print(f"   Similarity: {word['similarity']} | Confidence: {word['confidence']}")
            print()

        if result['unidentified_segments']:
            print("UNIDENTIFIED SEGMENTS:")
            for seg in result['unidentified_segments']:
                print(f"  '{seg['segment']}' at position {seg['position']}")


# Usage:
if __name__ == "__main__":
    # Initialize with your Excel file
    analyzer = TamilInscriptionAnalyzer("/content/drive/MyDrive/SEMESTER 5/Tamil Inscription Reader/Meanings/NLP Dataset (1 to 40 and 101 to 140).xlsx")

    # Analyze continuous Tamil text
    continuous_text = "ஊசுநிறுயிறுசிநழிறுபிநிஐெனயிழிழி"
    result = analyzer.analyze_inscription(continuous_text)

    # Print results
    analyzer.print_results(result)

Loaded 1531 dictionary entries
Analyzing text: ஊசுநிறுயிறுசிநழிறுபிநிஐெனயிழிழி...

Original text: ஊசுநிறுயிறுசிநழிறுபிநிஐெனயிழிழி
Identification rate: 100.00%
Identified words: 12

IDENTIFIED WORDS:
1. 'ஊசு' → ஏசு [old]
   Old: ஏசு | Modern: ஏசென்னேவல்
   English: revile
   Similarity: 0.667 | Confidence: medium

2. 'நி' → அநாதி [old]
   Old: அநாதி | Modern: கடவுள்
   English: the Deity
   Similarity: 0.511 | Confidence: low

3. 'று' → ஆறு [modern]
   Old: ஒலியல் | Modern: ஆறு
   English: a river
   Similarity: 0.967 | Confidence: high

4. 'யிறு' → கயிறு [modern]
   Old: கச்சை | Modern: கயிறு
   English: a rope
   Similarity: 0.98 | Confidence: high

5. 'சி' → அசி [old]
   Old: அசி | Modern: அவமதிச்சிரிப்பு
   English: a sneer
   Similarity: 0.967 | Confidence: high

6. 'நழி' → கழி [old]
   Old: கழி | Modern: உப்பளம்
   English: saltpans
   Similarity: 0.667 | Confidence: medium

7. 'று' → ஆறு [modern]
   Old: ஒலியல் | Modern: ஆறு
   English: a river
   Similarity: 0.967 | Confidence: 